In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Literal
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field


In [ ]:
load_dotenv()

In [ ]:
generator_model = ChatGroq(model="openai/gpt-oss-120b")
optimizer_model = ChatGroq(model="openai/gpt-oss-120b")
evaluator_model = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
class EvaluatorSchema(BaseModel):
    
    feedback: str = Field(description="Detailed feedback for the joke")
    evaluation: Literal['approved', 'needs_improvement'] = Field(description="Evaluation result of the generated tweet")


In [ ]:
evaluator_model_with_structured_output = evaluator_model.with_structured_output(EvaluatorSchema)

In [ ]:
# Defining State

class TweetState(TypedDict):
    
    topic: str
    tweet: str
    evaluation:  Literal['approved', 'needs_improvement']
    feedback: str
    iteration: int
    max_iteration: int

In [ ]:
graph = StateGraph(TweetState)


In [ ]:
def generate_tweet(state: TweetState):

    messages = [
        SystemMessage(
            content="""
                You are a witty internet comedian who specializes in creating genuinely funny, sarcastic, clever, and relatable posts for X (formerly Twitter).

                Your task is to generate exactly ONE funny X post based on the topic provided by the user.

                Follow these rules:
                1. Find the funniest angle of the topic instead of describing it literally.
                2. Use sarcasm, irony, unexpected twists, clever observations, absurd humor, or relatable situations whenever appropriate.
                3. Start with an attention-grabbing hook and finish with a memorable punchline when suitable.
                4. Make the post sound like a real person posting on X, not an AI, marketer, or corporate social media manager.
                5. Avoid generic jokes, predictable punchlines, clichés, forced humor, and motivational quotes.
                6. If the topic is technical, use developer humor or technology-related references only when they naturally fit.
                7. Keep the post concise and within X's standard 280-character limit, including spaces and punctuation.
                8. Do not invent facts, statistics, or personal experiences.
                9. Use emojis and hashtags only when they genuinely improve the humor.
                10. Prioritize originality, natural language, and comedic impact over unnecessary complexity.

                Output requirements:
                - Generate exactly ONE post.
                - Return only the final post as a plain string.
                - Do not include explanations, introductions, headings, character counts, or alternative versions.
                - Do not explain the joke.
                """
),
        HumanMessage(
            content=f"""
                Generate one genuinely funny X post based on the following topic:

                Topic: {state['topic']}

                Make it witty, entertaining, and relatable while following all the instructions provided in the system message.

                Return only the final post, ready to publish on X."""
        ),
    ]
    
    response = generator_model.invoke(messages)
    
    return {"tweet": response.content}


In [ ]:
def evaluate_tweet(state: TweetState):
    
    messages =[
        SystemMessage(content="""
            You are a strict comedy critic specializing in funny, witty, sarcastic, and relatable posts for X (formerly Twitter).

            Your task is to evaluate a generated X post and determine whether it is funny and good enough to publish.

            Evaluate the post using these criteria:
            1. Humor: Is it genuinely funny, or does it merely attempt to be funny?
            2. Originality: Is the joke creative and unexpected rather than generic or predictable?
            3. Relatability: Can the intended audience relate to the joke?
            4. Punchline: Is the punchline clever, sharp, and effective?
            5. Topic relevance: Does the humor naturally connect to the original topic?
            6. Conciseness: Is the post clear, natural, and within X's 280-character limit?

            Decision rules:
            - "approved": The post is genuinely funny, sufficiently original, relevant to the topic, and ready to publish without significant changes.
            - "needs_improvement": The post is unfunny, generic, predictable, forced, unclear, irrelevant, too long, or requires meaningful improvement.

            Be honest and critical. Do not approve a post merely because it is grammatically correct. Do not confuse sarcasm with genuine humor or claim that a post will go viral.

            Feedback rules:
            - Explain the main strengths and weaknesses of the post.
            - For an approved post, briefly explain why its humor works.
            - For a post needing improvement, explain the specific problems and suggest how to make it funnier.
            - When improvement is needed, include one improved version of the post within the feedback. Keep it within 280 characters.
            - Keep the feedback concise, specific, and actionable.

            Return the result according to the provided structured output schema:
            - feedback: str
            - decision: Literal["approved", "needs_improvement"]

            The decision must be exactly "approved" or "needs_improvement". Do not return any additional fields."""),
        HumanMessage(content=f"""
            Evaluate the following X post.

            Original Topic:
            {state['topic']}

            Generated X Post:
            {state['tweet']}

            Evaluate its humor, originality, relatability, punchline, topic relevance, and conciseness according to the system instructions.

            Return the feedback and decision using the required structured output schema.
            """)

    ]
    
    response = evaluator_model_with_structured_output.invoke(messages)
    
    return {
        "feedback": response.feedback,
        "evaluation": response.evaluation
    }

In [ ]:
def optimize_tweet(state: TweetState):
    
    messages = [
        SystemMessage(content="""
            You are an expert comedy writer specializing in optimizing funny, witty, sarcastic, and relatable posts for X (formerly Twitter).

            Your task is to improve an existing X post using the evaluator's feedback.

            Follow these instructions:

            Carefully analyze the original topic, the generated post, and the evaluator's feedback.

            Identify the specific weaknesses in the original post, such as weak humor, predictable punchlines, lack of originality, poor relatability, or awkward wording.

            Address the evaluator's criticism while preserving the original topic and core idea whenever possible.

            Make the humor sharper, more creative, natural, and entertaining.

            Use sarcasm, irony, unexpected twists, clever observations, or absurd humor when appropriate.

            Avoid generic AI-style jokes, forced humor, clichés, and predictable punchlines.

            Do not blindly follow feedback if doing so would make the joke less funny. Use your own judgment.

            Keep the optimized post within X's standard 280-character limit, including spaces and punctuation.

            Do not invent personal experiences, statistics, or factual claims.

            Generate exactly ONE optimized post.

            Output requirements:

            Return only the final optimized X post as a plain string.

            Do not include explanations, headings, quotation marks, character counts, or alternative versions.

            Do not evaluate the post or discuss the changes you made.

            Golden rule: Do not merely rephrase the original post. Fix its actual weaknesses and make the joke genuinely funnier while keeping it relevant to the topic."""),
        HumanMessage(content=f"""
            Optimize the following X post based on the evaluator's feedback.

            Original Topic:
            {state['topic']}

            Original X Post:
            {state['tweet']}

            Evaluator Feedback:
            {state['feedback']}

            Generate one improved version that addresses the identified weaknesses, strengthens the humor, and remains relevant to the original topic.

            Return only the final optimized post as a plain string.        
            """)
    ]
    
    response = optimizer_model.invoke(messages).content
    
    return {
        "tweet" : response,
        "iteration" : state['iteration'] + 1
    }

In [ ]:
def route_evaluation(state: TweetState):
    
    if state['evaluation'] == "approved" or state['iteration'] >= state['max_iteration']:
        return "approved"
    else:
        return "needs_improvement"

In [ ]:
graph.add_node("generate_tweet", generate_tweet)
graph.add_node("evaluate_tweet", evaluate_tweet)
graph.add_node("optimize_tweet", optimize_tweet)


graph.add_edge(START, "generate_tweet")
graph.add_edge("generate_tweet", "evaluate_tweet")
graph.add_conditional_edges("evaluate_tweet" , route_evaluation, {'approved': END, 'needs_improvement': "optimize_tweet"})
graph.add_edge("optimize_tweet", "evaluate_tweet")

workflow = graph.compile()
workflow

In [ ]:
initial_state= {
    "topic": "Cricket",
    "max_iteration":5,
    "iteration": 1
}

result = workflow.invoke(initial_state)

result